# Phase 3a — Feature Engineering + ALS/Item-CF Candidate Generation
Input: data/processed/{train,val}.parquet (from Phase 2)

## Cell 1: Load splits + cache the leakage-safe category snapshot
Reuse `T1` from Phase 2. The `categoryid` snapshot (timestamp < T1) is expensive to rebuild (two ~450MB CSVs), so cache it to parquet once and reuse across this notebook and `03b_ranking.ipynb`.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"  # implicit/ALS warns OpenBLAS's own threadpool causes severe slowdowns

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import sys
sys.path.insert(0, "..")
from src.baseline import compute_popularity_score, load_item_category_snapshot

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

train = pd.read_parquet(PROCESSED_DIR / "train.parquet")
val = pd.read_parquet(PROCESSED_DIR / "val.parquet")

with open(PROCESSED_DIR / "split_dates.json") as f:
    split_dates = json.load(f)
T1 = split_dates["T1_ms"]
T2 = split_dates["T2_ms"]

print("train:", train.shape, " val:", val.shape)
print("T1:", pd.to_datetime(T1, unit="ms"), " T2:", pd.to_datetime(T2, unit="ms"))

cat_cache_path = PROCESSED_DIR / "item_category_train.parquet"
if cat_cache_path.exists():
    item_category = pd.read_parquet(cat_cache_path).set_index("itemid")["categoryid"]
    print(f"\nloaded cached item_category snapshot: {len(item_category):,} items")
else:
    t0 = time.time()
    item_category = load_item_category_snapshot(T1, RAW_DIR)
    item_category.rename("categoryid").reset_index().to_parquet(cat_cache_path, index=False)
    print(f"\nbuilt + cached item_category snapshot: {len(item_category):,} items ({time.time()-t0:.1f}s)")

train: (2204512, 6)  val: (275564, 6)
T1: 2015-08-18 04:23:20.445000  T2: 2015-09-02 17:49:14.032000

loaded cached item_category snapshot: 411,807 items


## Cell 2: USER features (train only)
`user_total_views/carts/purchases`, `user_unique_items_viewed`, `user_top_category` (mode of viewed-item categories), `user_days_since_last_event` (recency relative to T1), `user_interaction_span_days`.

In [2]:
DAY_MS = 1000 * 60 * 60 * 24

event_counts = (
    train.pivot_table(index="visitorid", columns="event", values="itemid", aggfunc="count", fill_value=0)
)
for col in ["view", "addtocart", "transaction"]:
    if col not in event_counts.columns:
        event_counts[col] = 0

user_features = event_counts.rename(columns={
    "view": "user_total_views", "addtocart": "user_total_carts", "transaction": "user_total_purchases",
})[["user_total_views", "user_total_carts", "user_total_purchases"]]

user_features["user_unique_items_viewed"] = (
    train[train["event"] == "view"].groupby("visitorid")["itemid"].nunique()
).reindex(user_features.index, fill_value=0)

view_events = train[train["event"] == "view"].copy()
view_events["categoryid"] = view_events["itemid"].map(item_category)
view_cat = view_events.dropna(subset=["categoryid"])
user_top_category = view_cat.groupby("visitorid")["categoryid"].agg(lambda s: s.value_counts().idxmax())
user_features["user_top_category"] = user_top_category

last_event = train.groupby("visitorid")["timestamp"].max()
first_event = train.groupby("visitorid")["timestamp"].min()
user_features["user_days_since_last_event"] = (T1 - last_event) / DAY_MS
user_features["user_interaction_span_days"] = (last_event - first_event) / DAY_MS

print("user_features shape:", user_features.shape)
print("\nnull rate per column:\n", user_features.isnull().mean())
print("\ndescribe:\n", user_features.describe())
print("\nsample:\n", user_features.sample(5, random_state=42))

user_features shape: (1123765, 7)

null rate per column:
 event
user_total_views              0.000000
user_total_carts              0.000000
user_total_purchases          0.000000
user_unique_items_viewed      0.000000
user_top_category             0.125922
user_days_since_last_event    0.000000
user_interaction_span_days    0.000000
dtype: float64



describe:
 event  user_total_views  user_total_carts  user_total_purchases  \
count      1.123765e+06      1.123765e+06          1.123765e+06   
mean       1.897156e+00      4.866676e-02          1.589656e-02   
std        1.036330e+01      1.100999e+00          7.615778e-01   
min        0.000000e+00      0.000000e+00          0.000000e+00   
25%        1.000000e+00      0.000000e+00          0.000000e+00   
50%        1.000000e+00      0.000000e+00          0.000000e+00   
75%        2.000000e+00      0.000000e+00          0.000000e+00   
max        5.465000e+03      6.100000e+02          4.820000e+02   

event  user_unique_items_viewed  user_days_since_last_event  \
count              1.123765e+06                1.123765e+06   
mean               1.515748e+00                5.185761e+01   
std                6.690555e+00                3.064622e+01   
min                0.000000e+00                2.235648e-04   
25%                1.000000e+00                2.526295e+01   
50%   

## Cell 3: ITEM features (train only)
`item_total_views/carts/purchases`, `item_cart_rate` (carts/views, only where views>10), `item_purchase_rate` (purchases/carts, only where carts>5), `item_category`, `item_days_since_first_seen`, `item_popularity`.

In [3]:
item_event_counts = (
    train.pivot_table(index="itemid", columns="event", values="visitorid", aggfunc="count", fill_value=0)
)
for col in ["view", "addtocart", "transaction"]:
    if col not in item_event_counts.columns:
        item_event_counts[col] = 0

item_features = item_event_counts.rename(columns={
    "view": "item_total_views", "addtocart": "item_total_carts", "transaction": "item_total_purchases",
})[["item_total_views", "item_total_carts", "item_total_purchases"]]

item_features["item_cart_rate"] = np.where(
    item_features["item_total_views"] > 10,
    item_features["item_total_carts"] / item_features["item_total_views"],
    np.nan,
)
item_features["item_purchase_rate"] = np.where(
    item_features["item_total_carts"] > 5,
    item_features["item_total_purchases"] / item_features["item_total_carts"],
    np.nan,
)

item_features["item_category"] = item_category.reindex(item_features.index)

first_seen = train.groupby("itemid")["timestamp"].min()
item_features["item_days_since_first_seen"] = (T1 - first_seen) / DAY_MS

popularity_score = compute_popularity_score(train)
item_features["item_popularity"] = popularity_score.reindex(item_features.index, fill_value=0)

print("item_features shape:", item_features.shape)
null_rates = item_features.isnull().mean()
print("\nnull rate per column:\n", null_rates)

print("\n--- null-rate documentation (roadmap requires explanation for any column > 20%) ---")
print("item_cart_rate      : 80.5% null -- undefined below the views>10 activity threshold (most")
print("                       items are long-tail, per Phase 1 EDA). Left as NaN for XGBoost.")
print("item_purchase_rate  : 99.1% null -- undefined below the carts>5 threshold; purchases are")
print("                       the rarest event type (0.81% of all events, Phase 1 Q1). Left as NaN.")
print(f"item_category       : {null_rates['item_category']:.1%} null -- these items have no")
print("                       categoryid property record with timestamp < T1 at all (not merely a")
print("                       time-varying value we're excluding for leakage -- the item simply has")
print("                       no such record yet, e.g. newly-listed items with a sparse property")
print("                       trail). Left as NaN; category_affinity (Cell 4) will also be NaN for")
print("                       these items rather than defaulting to False.")

print("\ndescribe:\n", item_features.describe())
print("\nsample:\n", item_features.sample(5, random_state=42))

item_features shape: (212915, 8)

null rate per column:
 event
item_total_views              0.000000
item_total_carts              0.000000
item_total_purchases          0.000000
item_cart_rate                0.804537
item_purchase_rate            0.990837
item_category                 0.209563
item_days_since_first_seen    0.000000
item_popularity               0.000000
dtype: float64

--- null-rate documentation (roadmap requires explanation for any column > 20%) ---
item_cart_rate      : 80.5% null -- undefined below the views>10 activity threshold (most
                       items are long-tail, per Phase 1 EDA). Left as NaN for XGBoost.
item_purchase_rate  : 99.1% null -- undefined below the carts>5 threshold; purchases are
                       the rarest event type (0.81% of all events, Phase 1 Q1). Left as NaN.
item_category       : 21.0% null -- these items have no
                       categoryid property record with timestamp < T1 at all (not merely a
                   

## Cell 4: USER-ITEM features (train only)
`user_item_views/carts/purchases`, `user_item_days_since_last_interaction`, `category_affinity` (does the item's category match the user's top viewed category).

In [4]:
ui_pivot = (
    train.pivot_table(index=["visitorid", "itemid"], columns="event", values="timestamp", aggfunc="count", fill_value=0)
    .reset_index()
)
for col in ["view", "addtocart", "transaction"]:
    if col not in ui_pivot.columns:
        ui_pivot[col] = 0

user_item_features = ui_pivot.rename(columns={
    "view": "user_item_views", "addtocart": "user_item_carts", "transaction": "user_item_purchases",
})[["visitorid", "itemid", "user_item_views", "user_item_carts", "user_item_purchases"]]

ui_last = train.groupby(["visitorid", "itemid"])["timestamp"].max().reset_index(name="last_ts")
user_item_features = user_item_features.merge(ui_last, on=["visitorid", "itemid"])
user_item_features["user_item_days_since_last_interaction"] = (T1 - user_item_features["last_ts"]) / DAY_MS
user_item_features = user_item_features.drop(columns="last_ts")

item_cat_for_ui = user_item_features["itemid"].map(item_category)
user_cat_for_ui = user_item_features["visitorid"].map(user_top_category)
both_known = item_cat_for_ui.notna() & user_cat_for_ui.notna()
category_affinity = pd.Series(np.nan, index=user_item_features.index)
category_affinity[both_known] = (item_cat_for_ui[both_known] == user_cat_for_ui[both_known]).astype(float)
user_item_features["category_affinity"] = category_affinity

print("user_item_features shape:", user_item_features.shape)
print("\nnull rate per column:\n", user_item_features.isnull().mean())
print(f"\ncategory_affinity: {(category_affinity == 1.0).sum():,} matches, "
      f"{(category_affinity == 0.0).sum():,} non-matches, {category_affinity.isnull().sum():,} unknown "
      f"({category_affinity.isnull().mean():.1%})")
print("\nsample:\n", user_item_features.sample(5, random_state=42))

user_item_features shape: (1713171, 7)

null rate per column:
 visitorid                                0.000000
itemid                                   0.000000
user_item_views                          0.000000
user_item_carts                          0.000000
user_item_purchases                      0.000000
user_item_days_since_last_interaction    0.000000
category_affinity                        0.106142
dtype: float64

category_affinity: 1,298,173 matches, 233,159 non-matches, 181,839 unknown (10.6%)

sample:
          visitorid  itemid  user_item_views  user_item_carts  \
1648073    1352944  386401                1                0   
812756      667303  191320                1                0   
1074652     881917   68595                1                0   
340950      281319  228831                1                0   
734128      601497  110879                1                0   

         user_item_purchases  user_item_days_since_last_interaction  \
1648073               

## Cell 5: Leakage check (mandatory)
Assert every feature source is train-only (timestamp < T1), and cache the three feature tables for reuse in `03b_ranking.ipynb`.

In [5]:
assert train["timestamp"].max() < T1, "train contains events >= T1"
assert (user_features["user_days_since_last_event"] >= 0).all(), "negative user recency -> leakage"
assert (item_features["item_days_since_first_seen"] >= 0).all(), "negative item recency -> leakage"
assert (user_item_features["user_item_days_since_last_interaction"] >= 0).all(), "negative user-item recency -> leakage"
print("leakage check: all recency features non-negative relative to T1 -> PASSED")

print("\nfeature matrix shapes:")
print("  user_features:     ", user_features.shape)
print("  item_features:     ", item_features.shape)
print("  user_item_features:", user_item_features.shape)

user_features.reset_index().to_parquet(PROCESSED_DIR / "user_features.parquet", index=False)
item_features.reset_index().to_parquet(PROCESSED_DIR / "item_features.parquet", index=False)
user_item_features.to_parquet(PROCESSED_DIR / "user_item_features.parquet", index=False)
print("\nsaved user_features.parquet, item_features.parquet, user_item_features.parquet")

leakage check: all recency features non-negative relative to T1 -> PASSED

feature matrix shapes:
  user_features:      (1123765, 7)
  item_features:      (212915, 8)
  user_item_features: (1713171, 7)



saved user_features.parquet, item_features.parquet, user_item_features.parquet


## Stage 1 — Candidate Generation: ALS + Item-CF
### Cell 6: Build ALS model
`factors=50, iterations=20, regularization=0.1` — small factors keep this fast and interpretable on a sparse, mostly-thin-history dataset (median 1 interaction/user) without overfitting.

In [6]:
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

# Confidence weight per (user,item): same weighting scheme as the popularity score.
ui_weight = (
    user_item_features["user_item_views"] * 1
    + user_item_features["user_item_carts"] * 2
    + user_item_features["user_item_purchases"] * 4
).astype(np.float32)

users_unique = user_item_features["visitorid"].unique()
items_unique = user_item_features["itemid"].unique()
user_to_idx = {u: i for i, u in enumerate(users_unique)}
item_to_idx = {it: i for i, it in enumerate(items_unique)}
idx_to_item = np.array(items_unique)
idx_to_user = np.array(users_unique)

row = user_item_features["visitorid"].map(user_to_idx).values
col = user_item_features["itemid"].map(item_to_idx).values

user_item_csr = csr_matrix((ui_weight.values, (row, col)), shape=(len(users_unique), len(items_unique)))
print("user_item_csr shape:", user_item_csr.shape, " nnz:", user_item_csr.nnz)

als_model = AlternatingLeastSquares(factors=50, iterations=20, regularization=0.1, random_state=42)
t0 = time.time()
als_model.fit(user_item_csr)
als_train_time = time.time() - t0
print(f"\nALS trained: factors=50, iterations=20, regularization=0.1")
print(f"training time: {als_train_time:.1f}s")
print("user_factors shape:", als_model.user_factors.shape)
print("item_factors shape:", als_model.item_factors.shape)

user_item_csr shape: (1123765, 212915)  nnz: 1713171


  0%|          | 0/20 [00:00<?, ?it/s]


ALS trained: factors=50, iterations=20, regularization=0.1
training time: 104.0s
user_factors shape: (1123765, 50)
item_factors shape: (212915, 50)


### Cell 7: Build Item-CF
Item-item cosine similarity from the train user-item matrix, top-20 similar items per item — captures local "viewed X, also viewed Y" behavior that ALS's global factors can miss.

In [7]:
from implicit.nearest_neighbours import CosineRecommender

itemcf_model = CosineRecommender(K=20)
t0 = time.time()
itemcf_model.fit(user_item_csr)
itemcf_train_time = time.time() - t0
print(f"Item-CF (cosine, K=20) trained in {itemcf_train_time:.1f}s")
print("similarity matrix shape:", itemcf_model.similarity.shape, " nnz:", itemcf_model.similarity.nnz)

print("\nsample similar items for 3 items:")
for it in idx_to_item[:3]:
    idx = item_to_idx[it]
    ids, scores = itemcf_model.similar_items(idx, N=5)
    similar_itemids = idx_to_item[ids]
    pairs = list(zip(similar_itemids.tolist(), np.round(scores, 3).tolist()))
    print(f"  item {it}: {pairs}")

C:\Users\sriva\AppData\Local\Programs\Python\Python313\Lib\site-packages\implicit\utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.019613981246948242 seconds
  warnings.warn(


  0%|          | 0/212915 [00:00<?, ?it/s]

Item-CF (cosine, K=20) trained in 1.6s
similarity matrix shape: (212915, 212915)  nnz: 1615532

sample similar items for 3 items:
  item 72028: [(72028, 1.0), (963, 0.341), (128782, 0.149), (247808, 0.135), (276885, 0.109)]
  item 216305: [(216305, 1.0), (29940, 0.205), (441734, 0.156), (172103, 0.148), (325215, 0.141)]
  item 259884: [(259884, 1.0), (441734, 0.242), (342264, 0.179), (78268, 0.138), (29940, 0.135)]


### Cell 8: Generate candidate pool
`candidate_score(u,i) = alpha*als_score + (1-alpha)*itemcf_score`, alpha in {0.5, 0.6, 0.7} tuned on val to maximize candidate Recall@50 (scores used as-is per the roadmap's formula, no extra normalization step). Retrieve top-50 candidates per user (not the final top-10), from the union of each model's own top-50 — this is candidate generation, not the final ranking decision.

**Seen-item filtering, corrected after Cell 1 of `03b_ranking.ipynb` exposed the consequence of getting this wrong:** an earlier version of this cell filtered *all* train-interacted items via implicit's `filter_already_liked_items=True`, reasoning that Item-CF's self-similarity would otherwise just re-surface a user's own history. That filter also removed every merely-*viewed* item — which turned out to be exactly the items most likely to convert to addtocart/purchase on a return visit (browse-then-buy is the norm here). With that filter on, only 46 of 762,000 candidate rows were positive labels, far too sparse to train anything. Filtering now matches the roadmap's literal spec: retrieval keeps viewed/carted items and excludes only items the user already **purchased** in train, applied post-hoc (not via the library's blanket filter).

Candidate generation only covers users present in the train ALS/Item-CF fit (warm users) — true cold-start users (no train history) get no personalized candidates and rely entirely on the popularity fallback, per the roadmap's failure-mode design.

In [8]:
from src.recommender import generate_candidates
from src.evaluate import recall_at_k

val_users_set = set(val["visitorid"].unique())
warm_val_mask = np.isin(users_unique, np.array(list(val_users_set)))
warm_val_user_idx = np.nonzero(warm_val_mask)[0]
print(f"train users also present in val (warm): {len(warm_val_user_idx):,} / {len(users_unique):,} train users")
print(f"  = {len(warm_val_user_idx) / len(val_users_set):.1%} of {len(val_users_set):,} val users")

# Items each user already purchased in train -> excluded from their candidates post-hoc
train_purchases = train[train["event"] == "transaction"][["visitorid", "itemid"]].drop_duplicates()
train_purchases["user_idx"] = train_purchases["visitorid"].map(user_to_idx)
train_purchases["item_idx"] = train_purchases["itemid"].map(item_to_idx)
purchased_item_idx = train_purchases.groupby("user_idx")["item_idx"].apply(set).to_dict()
print(f"users with >=1 train purchase to exclude: {len(purchased_item_idx):,}")

t0 = time.time()
candidate_pool = generate_candidates(
    als_model, itemcf_model, user_item_csr, warm_val_user_idx, purchased_item_idx, candidate_n=50,
)
print(f"\ncandidate retrieval done in {time.time()-t0:.1f}s, pool shape: {candidate_pool.shape}")

candidate_pool["itemid"] = idx_to_item[candidate_pool["item_idx"].values]
candidate_pool["visitorid"] = idx_to_user[candidate_pool["user_idx"].values]

print("\ncandidate_source distribution:\n", candidate_pool["candidate_source"].value_counts())
per_user_counts = candidate_pool.groupby("visitorid").size()
print(f"\ncandidates per user: mean={per_user_counts.mean():.1f}, min={per_user_counts.min()}, max={per_user_counts.max()}")

# --- alpha tuning: maximize candidate Recall@50 on val ---
relevant_any_val = val.groupby("visitorid")["itemid"].apply(set).to_dict()
eligible_visitorids = set(candidate_pool["visitorid"].unique())
relevant_eligible = {u: r for u, r in relevant_any_val.items() if u in eligible_visitorids}
print(f"\nusers eligible for alpha tuning (warm + in val): {len(relevant_eligible):,}")

best_alpha, best_recall = None, -1
for alpha in [0.5, 0.6, 0.7]:
    scored = candidate_pool.assign(blended=alpha * candidate_pool["als_score"] + (1 - alpha) * candidate_pool["itemcf_score"])
    top50 = scored.sort_values(["visitorid", "blended"], ascending=[True, False]).groupby("visitorid").head(50)
    recs_by_user = top50.groupby("visitorid")["itemid"].apply(list).to_dict()
    recall, n = recall_at_k(recs_by_user, relevant_eligible)
    print(f"alpha={alpha}: candidate Recall@50={recall:.4f}  (n_users={n:,})")
    if recall > best_recall:
        best_alpha, best_recall = alpha, recall

print(f"\nbest alpha: {best_alpha}  (Recall@50={best_recall:.4f})")

train users also present in val (warm): 15,240 / 1,123,765 train users
  = 9.7% of 156,954 val users


users with >=1 train purchase to exclude: 9,343



candidate retrieval done in 12.7s, pool shape: (1120415, 5)

candidate_source distribution:
 candidate_source
als       744123
itemcf    358437
both       17855
Name: count, dtype: int64

candidates per user: mean=73.5, min=50, max=100



users eligible for alpha tuning (warm + in val): 15,240


alpha=0.5: candidate Recall@50=0.2103  (n_users=15,240)


alpha=0.6: candidate Recall@50=0.2105  (n_users=15,240)


alpha=0.7: candidate Recall@50=0.2108  (n_users=15,240)

best alpha: 0.7  (Recall@50=0.2108)


In [9]:
candidate_pool["candidate_score"] = best_alpha * candidate_pool["als_score"] + (1 - best_alpha) * candidate_pool["itemcf_score"]

final_candidates = (
    candidate_pool.sort_values(["visitorid", "candidate_score"], ascending=[True, False])
    .groupby("visitorid").head(50)
    .copy()
)
final_candidates["rank"] = final_candidates.groupby("visitorid").cumcount() + 1
final_candidates = final_candidates[
    ["visitorid", "itemid", "als_score", "itemcf_score", "candidate_score", "candidate_source", "rank"]
]
final_candidates.to_parquet(PROCESSED_DIR / "candidates.parquet", index=False)

print(f"saved candidates.parquet: {final_candidates.shape}")
print("mean candidates per user:", final_candidates.groupby("visitorid").size().mean())

print("\nsample candidates for 3 users:")
for uid in final_candidates["visitorid"].unique()[:3]:
    sub = final_candidates[final_candidates["visitorid"] == uid].head(5)
    print(f"\n  user {uid}:")
    print(sub[["itemid", "candidate_score", "candidate_source", "rank"]].to_string(index=False))

saved candidates.parquet: (762000, 7)
mean candidates per user: 50.0

sample candidates for 3 users:

  user 155:
 itemid  candidate_score candidate_source  rank
 134620         0.600000           itemcf     1
 123027         0.351772           itemcf     2
  91454         0.208700           itemcf     3
 462664         0.123956           itemcf     4
 216804         0.104350           itemcf     5

  user 162:
 itemid  candidate_score candidate_source  rank
 390093         0.304435             both     1
   1152         0.300000           itemcf     2
 248862         0.300000           itemcf     3
 305656         0.300000           itemcf     4
    678         0.105263           itemcf     5

  user 295:
 itemid  candidate_score candidate_source  rank
 445817         0.300000           itemcf     1
  59625         0.018325           itemcf     2
 400940         0.014174           itemcf     3
 211796         0.013927           itemcf     4
 303960         0.006904           itemcf   

## Cell 9: Persist model artifacts
Save the trained ALS model, Item-CF model, and the user/item id mappings (needed to translate between raw ids and matrix indices) for reuse in `03b_ranking.ipynb`, Phase 4, and Phase 5's DVC `train` stage.

In [10]:
import pickle

with open(PROCESSED_DIR / "als_model.pkl", "wb") as f:
    pickle.dump(als_model, f)
with open(PROCESSED_DIR / "itemcf_matrix.pkl", "wb") as f:
    pickle.dump(itemcf_model, f)
with open(PROCESSED_DIR / "id_mappings.pkl", "wb") as f:
    pickle.dump({
        "user_to_idx": user_to_idx, "item_to_idx": item_to_idx,
        "idx_to_user": idx_to_user, "idx_to_item": idx_to_item,
        "best_alpha": best_alpha,
    }, f)

for name in ["als_model.pkl", "itemcf_matrix.pkl", "id_mappings.pkl"]:
    size_mb = (PROCESSED_DIR / name).stat().st_size / 1e6
    print(f"saved {name}: {size_mb:.1f} MB")

saved als_model.pkl: 267.3 MB
saved itemcf_matrix.pkl: 20.2 MB
saved id_mappings.pkl: 42.5 MB
